In [ ]:
# Instala o Unsloth diretamente do PyPI (evita o erro do git clone)
!pip install unsloth




     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 M

In [ ]:
import pandas as pd

# Carrega o dataset de coerência de processo
dataset_url = "https://huggingface.co/datasets/jotabpmn/bpmn-model-ia-generative/raw/main/process_coherence_dataset.csv"
coherence_df = pd.read_csv(dataset_url)

print("Dataset 'process_coherence_dataset.csv' carregado com sucesso!")
display(coherence_df.head())

Dataset 'process_coherence_dataset.csv' carregado com sucesso!


,process_name,text_description,bpmn_xml
0,Client Acquisition,After management indicates that new clients ar...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<bpmn:..."
1,Credit,The credit company receives the credit informa...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<bpmn:..."
2,Dispatching,"If goods shall be shipped, the secretary clari...","<?xml version=""1.0"" encoding=""UTF-8""?>\n<bpmn:..."
3,Hospital,The examination process can be summarised as f...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<bpmn2..."
4,Hotel,The Evanstonian is an upscale independent hote...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<bpmn:..."


In [ ]:
import os
import torch

# Activates dynamic segment management to avoid fragmentation out-of-memory errors
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from unsloth import FastLanguageModel

# MAXIMUM SEQUENCE LENGTH
max_seq_length = 8192
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

print("🚀 Model and Tokenizer loaded successfully for inference!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


🚀 Model and Tokenizer loaded successfully for inference!


In [ ]:
import re
import unicodedata
import xml.etree.ElementTree as ET
from xml.dom import minidom
import lxml.etree as ET_lxml
import networkx as nx # Adicionado para corrigir o erro NameError: name 'nx' is not defined

def calculate_requirements_coverage_metric(reference_text, generated_xml, nlp_model, sentence_transformer_model):
    # Extrai o texto do modelo BPMN gerado
    texto_bpmn_gerado = extrair_texto_bpmn(generated_xml)

    # Gera embeddings para o texto de referência e o texto do BPMN gerado
    embeddings_texto_referencia = sentence_transformer_model.encode(reference_text, convert_to_tensor=True)
    embeddings_texto_bpmn_gerado = sentence_transformer_model.encode(texto_bpmn_gerado, convert_to_tensor=True)

    # Calcula a similaridade de cosseno
    similaridade_semantica = util.pytorch_cos_sim(embeddings_texto_referencia, embeddings_texto_bpmn_gerado).item() * 100
    return similaridade_semantica

def extrair_texto_bpmn(xml_string):
    # Extrai todos os nomes de elementos do BPMN para formar uma "descrição" do modelo
    root = ET_lxml.fromstring(xml_string.encode('utf-8'))
    nomes_elementos = []
    for elem in root.iter():
        if 'name' in elem.attrib:
            nomes_elementos.append(elem.attrib['name'])
    return ". ".join(nomes_elementos)


def calculate_ged_metric(generated_xml, reference_xml):
    # Converte os XMLs em grafos
    grafo_gerado_local = bpmn_to_networkx(generated_xml)
    grafo_referencia_local = bpmn_to_networkx(reference_xml)

    num_nodes_gerado = grafo_gerado_local.number_of_nodes()
    num_edges_gerado = grafo_gerado_local.number_of_edges()

    num_nodes_referencia = grafo_referencia_local.number_of_nodes()
    num_edges_referencia = grafo_referencia_local.number_of_edges()

    diff_nodes = abs(num_nodes_gerado - num_nodes_referencia)
    diff_edges = abs(num_edges_gerado - num_edges_referencia)

    ged_heuristico = diff_nodes + diff_edges

    total_elementos_gerado = num_nodes_gerado + num_edges_gerado
    total_elementos_referencia = num_nodes_referencia + num_edges_referencia

    if (total_elementos_gerado + total_elementos_referencia) > 0:
        max_possible_ged = total_elementos_gerado + total_elementos_referencia
        similaridade_ged = (1 - (ged_heuristico / max_possible_ged)) * 100
    else:
        similaridade_ged = 100.0 if ged_heuristico == 0 else 0.0

    return similaridade_ged

def validar_xml_sintaticamente(xml_string):
    try:
        # Tenta parsear o XML usando lxml para validação robusta
        # Deve-se codificar para bytes se a string contiver a declaração XML (e.g., <?xml ...?>)
        ET_lxml.fromstring(xml_string.encode('utf-8'))
        return 100.0 # XML é sintaticamente válido
    except ET_lxml.XMLSyntaxError:
        return 0.0 # XML não é sintaticamente válido

def run_syntactic_validation(xml_generated, xml_reference):
    validation_generated = validar_xml_sintaticamente(xml_generated)
    validation_reference = validar_xml_sintaticamente(xml_reference)
    return validation_generated, validation_reference




def sanitize_bpmn_id(value, prefix="Id"):
    value = str(value).strip()

    # Remove acentos
    value = unicodedata.normalize("NFKD", value)
    value = value.encode("ascii", "ignore").decode("ascii")

    # Mantém apenas letras, números e underscore
    value = re.sub(r"[^A-Za-z0-9_]", "_", value)

    # Compacta underscores repetidos
    value = re.sub(r"_+", "_", value).strip("_")

    # ID não deve começar com número
    if not value or not re.match(r"^[A-Za-z_]", value):
        value = f"{prefix}_{value}"

    return value


def make_unique_id(base_id, used_ids):
    candidate = base_id
    counter = 2

    while candidate in used_ids:
        candidate = f"{base_id}_{counter}"
        counter += 1

    used_ids.add(candidate)
    return candidate


def gerar_bpmn_definitivo(process_data):
    bpmn_namespace = "http://www.omg.org/spec/BPMN/20100524/MODEL"
    xsi_namespace = "http://www.w3.org/2001/XMLSchema-instance"
    bpmndi_namespace = "http://www.omg.org/spec/BPMN/20100524/DI"
    omgdc_namespace = "http://www.omg.org/spec/DD/20100524/DC"
    omgdi_namespace = "http://www.omg.org/spec/DD/20100524/DI"

    ET.register_namespace("bpmn", bpmn_namespace)
    ET.register_namespace("xsi", xsi_namespace)
    ET.register_namespace("bpmndi", bpmndi_namespace)
    ET.register_namespace("omgdc", omgdc_namespace)
    ET.register_namespace("omgdi", omgdi_namespace)

    used_ids = set()

    process_name = process_data["process_name"]

    definitions_id = make_unique_id(
        sanitize_bpmn_id(f"Definitions_{process_name}", "Definitions"),
        used_ids,
    )
    process_id = make_unique_id(
        sanitize_bpmn_id(f"Process_{process_name}", "Process"),
        used_ids,
    )

    element_id_map = {}
    for element_data in process_data["elements"]:
        original_id = element_data["id"]
        clean_id = sanitize_bpmn_id(original_id, "Element")
        element_id_map[original_id] = make_unique_id(clean_id, used_ids)

    flow_id_map = {}
    for flow_data in process_data["sequence_flows"]:
        original_id = flow_data["id"]
        clean_id = sanitize_bpmn_id(original_id, "Flow")
        flow_id_map[original_id] = make_unique_id(clean_id, used_ids)

    definitions = ET.Element(
        f"{{{bpmn_namespace}}}definitions",
        {
            "id": definitions_id,
            "targetNamespace": bpmn_namespace,
            f"{{{xsi_namespace}}}schemaLocation": f"{bpmn_namespace} BPMN20.xsd",
        },
    )

    process = ET.SubElement(
        definitions,
        f"{{{bpmn_namespace}}}process",
        {
            "id": process_id,
            "isExecutable": "false",
        },
    )

    element_coords = {}
    x_offset = 100
    y_offset = 100
    gap = 150
    row_height = 80
    current_y = y_offset

    for element_data in process_data["elements"]:
        elem_type = element_data["type"]
        elem_id = element_id_map[element_data["id"]]
        elem_name = element_data.get("name", "")

        if elem_type == "startEvent":
            ET.SubElement(
                process,
                f"{{{bpmn_namespace}}}startEvent",
                {"id": elem_id, "name": elem_name},
            )
            width, height = 36, 36

        elif elem_type == "endEvent":
            ET.SubElement(
                process,
                f"{{{bpmn_namespace}}}endEvent",
                {"id": elem_id, "name": elem_name},
            )
            width, height = 36, 36

        elif elem_type in ("userTask", "serviceTask"):
            ET.SubElement(
                process,
                f"{{{bpmn_namespace}}}{elem_type}",
                {"id": elem_id, "name": elem_name},
            )
            width, height = 100, 80

        elif elem_type == "exclusiveGateway":
            ET.SubElement(
                process,
                f"{{{bpmn_namespace}}}exclusiveGateway",
                {
                    "id": elem_id,
                    "name": elem_name,
                    "gatewayDirection": "Diverging",
                },
            )
            width, height = 50, 50

        else:
            continue

        element_coords[elem_id] = {
            "x": x_offset,
            "y": current_y,
            "width": width,
            "height": height,
        }

        x_offset += width + gap

        if x_offset > 800:
            x_offset = 100
            current_y += row_height + gap

    for flow_data in process_data["sequence_flows"]:
        source_original_id = flow_data["from"]
        target_original_id = flow_data["to"]

        if source_original_id not in element_id_map:
            raise ValueError(
                f"Fluxo '{flow_data['id']}' aponta para origem inexistente: {source_original_id}"
            )

        if target_original_id not in element_id_map:
            raise ValueError(
                f"Fluxo '{flow_data['id']}' aponta para destino inexistente: {target_original_id}"
            )

        ET.SubElement(
            process,
            f"{{{bpmn_namespace}}}sequenceFlow",
            {
                "id": flow_id_map[flow_data["id"]],
                "sourceRef": element_id_map[source_original_id],
                "targetRef": element_id_map[target_original_id],
            },
        )

    bpmndi_diagram = ET.SubElement(
        definitions,
        f"{{{bpmndi_namespace}}}BPMNDiagram",
        {"id": "BPMNDiagram_1"},
    )

    bpmndi_plane = ET.SubElement(
        bpmndi_diagram,
        f"{{{bpmndi_namespace}}}BPMNPlane",
        {
            "id": "BPMNPlane_1",
            "bpmnElement": process_id,
        },
    )

    for element_data in process_data["elements"]:
        elem_id = element_id_map[element_data["id"]]
        coords = element_coords.get(elem_id)

        if not coords:
            continue

        shape_id = make_unique_id(
            sanitize_bpmn_id(f"BPMNShape_{elem_id}", "BPMNShape"),
            used_ids,
        )

        shape = ET.SubElement(
            bpmndi_plane,
            f"{{{bpmndi_namespace}}}BPMNShape",
            {
                "id": shape_id,
                "bpmnElement": elem_id,
            },
        )

        ET.SubElement(
            shape,
            f"{{{omgdc_namespace}}}Bounds",
            {
                "x": str(coords["x"]),
                "y": str(coords["y"]),
                "width": str(coords["width"]),
                "height": str(coords["height"]),
            },
        )

    for flow_data in process_data["sequence_flows"]:
        flow_id = flow_id_map[flow_data["id"]]
        source_id = element_id_map[flow_data["from"]]
        target_id = element_id_map[flow_data["to"]]

        source_coords = element_coords.get(source_id)
        target_coords = element_coords.get(target_id)

        if not source_coords or not target_coords:
            continue

        edge_id = make_unique_id(
            sanitize_bpmn_id(f"BPMNEdge_{flow_id}", "BPMNEdge"),
            used_ids,
        )

        edge = ET.SubElement(
            bpmndi_plane,
            f"{{{bpmndi_namespace}}}BPMNEdge",
            {
                "id": edge_id,
                "bpmnElement": flow_id,
            },
        )

        ET.SubElement(
            edge,
            f"{{{omgdi_namespace}}}Waypoint",
            {
                "x": str(source_coords["x"] + source_coords["width"]),
                "y": str(source_coords["y"] + source_coords["height"] / 2),
            },
        )

        ET.SubElement(
            edge,
            f"{{{omgdi_namespace}}}Waypoint",
            {
                "x": str(target_coords["x"]),
                "y": str(target_coords["y"] + target_coords["height"] / 2),
            },
        )

    rough_string = ET.tostring(definitions, "utf-8")
    reparsed = minidom.parseString(rough_string)
    pretty_xml_as_string = reparsed.toprettyxml(indent="  ")

    pretty_xml_as_string = "\n".join(pretty_xml_as_string.split("\n")[1:])
    pretty_xml_as_string = "<?xml version=\"1.0\" encoding=\"UTF-8\"?>\n" + pretty_xml_as_string

    print(f"XML BPMN gerado com IDs compatíveis. Retornando string...")

    return pretty_xml_as_string


def calculate_structural_complexity_metric(generated_xml):
    # Converte o XML gerado em grafo
    grafo_gerado_local = bpmn_to_networkx(generated_xml)

    # Calcula a densidade do grafo
    # A densidade de um grafo direcionado é: E / (V * (V-1))
    num_nodes_local = grafo_gerado_local.number_of_nodes()
    num_edges_local = grafo_gerado_local.number_of_edges()

    if num_nodes_local > 1:
        densidade_grafo = num_edges_local / (num_nodes_local * (num_nodes_local - 1))
    else:
        densidade_grafo = 0.0

    return densidade_grafo * 100

def bpmn_to_networkx(xml_string):
    graph = nx.DiGraph()
    root = ET_lxml.fromstring(xml_string.encode('utf-8'))

    # List of BPMN element tags to consider as nodes
    bpmn_node_tags = [
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}task',
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}startEvent',
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}endEvent',
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}exclusiveGateway',
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}userTask',
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}serviceTask'
    ]

    # Add nodes
    for tag in bpmn_node_tags:
        for elem in root.iter(tag):
            graph.add_node(elem.attrib['id'], type=elem.tag.split('}')[-1], name=elem.attrib.get('name', ''))

    # Add edges
    for flow in root.iter('{http://www.omg.org/spec/BPMN/20100524/MODEL}sequenceFlow'):
        source_ref = flow.attrib.get('sourceRef')
        target_ref = flow.attrib.get('targetRef')
        if source_ref and target_ref:
            graph.add_edge(source_ref, target_ref)
    return graph

In [ ]:
import time

def validar_json_sintaticamente(json_string):
    """
    Tenta parsear a string JSON para validar sua sintaxe.
    Retorna True se o JSON for sintaticamente válido, False caso contrário.
    """
    try:
        json.loads(json_string)
        return True
    except json.JSONDecodeError:
        return False

def validar_bpmn_logicamente(process_data):
    """
    Valida a lógica BPMN de um dicionário de processo, verificando se os IDs de origem e destino
    dos sequence_flows existem na lista de elementos.
    Retorna True se for logicamente válido, False caso contrário.
    """
    elements_ids = set()
    for elem in process_data.get("elements", []):
        elem_id = elem.get("id")
        # Garante que o ID é um tipo hashable (string, int, float, bool)
        if not isinstance(elem_id, (str, int, float, bool)):
            print(f"❌ Erro de validação lógica BPMN: ID de elemento inválido (não-hashable): '{elem_id}'")
            return False
        elements_ids.add(elem_id)

    for flow_data in process_data.get("sequence_flows", []):
        source_id = flow_data.get("from")
        target_id = flow_data.get("to")

        # Garante que os IDs de origem e destino são tipos hashable
        if not isinstance(source_id, (str, int, float, bool)):
            print(f"❌ Erro de validação lógica BPMN: ID de origem de fluxo inválido (não-hashable): '{source_id}'")
            return False
        if not isinstance(target_id, (str, int, float, bool)):
            print(f"❌ Erro de validação lógica BPMN: ID de destino de fluxo inválido (não-hashable): '{target_id}'")
            return False

        if source_id not in elements_ids:
            print(f"❌ Erro de validação lógica BPMN: Fluxo '{flow_data.get('id', 'N/A')}' aponta para origem inexistente: '{source_id}'")
            return False
        if target_id not in elements_ids:
            print(f"❌ Erro de validação lógica BPMN: Fluxo '{flow_data.get('id', 'N/A')}' aponta para destino inexistente: '{target_id}'")
            return False
    return True


def run_model_until_valid_json(
    model,
    tokenizer,
    inputs,
    max_new_tokens_permitido,
    input_prompt_token_length,
    do_sample=True,
    repetition_penalty=1,
    use_cache=False,
    max_attempts=5
):

    retry_count = 0
    # Inicializa o tempo de geração fora do loop para ser retornado mesmo se não houver JSON válido
    generation_time = 0.0
    while retry_count < max_attempts:
        retry_count += 1
        print(f"\n⚡ Tentativa {retry_count} de {max_attempts}: Gerando texto...")

        # 1. Gerar o texto (chamando a função 'gerar texto' que foi refatorada)
        # A função generate_and_extract_json (cell 76b2c4d7) agora retorna apenas cleaned_output
        start_generate_time = time.time() # Record start time for generation
        outputs = model.generate(
          input_ids=inputs,
          max_new_tokens=max_new_tokens_permitido,
          do_sample=do_sample,
          repetition_penalty=repetition_penalty,
          use_cache=use_cache)
        end_generate_time = time.time() # Record end time for generation
        generation_time = end_generate_time - start_generate_time # Calcula o tempo de geração
        print(f"Tempo de geração do modelo: {generation_time:.2f} segundos")

        # Decodifica apenas os tokens gerados a partir do fim do input
        resultado_gerado = outputs[0][input_prompt_token_length:]
        cleaned_output = tokenizer.decode(resultado_gerado, skip_special_tokens=True).strip()

        # 2. Extrair o bloco JSON da string gerada
        codigo_json = ""
        json_start_marker = "```json"
        json_end_marker = "```"

        start_idx = cleaned_output.find(json_start_marker)
        end_idx = cleaned_output.find(json_end_marker, start_idx + len(json_start_marker))

        if start_idx != -1 and end_idx != -1:
            codigo_json = cleaned_output[start_idx + len(json_start_marker):end_idx].strip()
        else:
            print("Warning: Bloco Markdown JSON não encontrado. Tentando chaves estruturais.")
            first_brace = cleaned_output.find('{')
            last_brace = cleaned_output.rfind('}')
            if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
                codigo_json = cleaned_output[first_brace:last_brace+1].strip()

        # 3. Validar a sintaxe JSON
        if validar_json_sintaticamente(codigo_json):
            print(f"✅ JSON sintaticamente válido encontrado na tentativa {retry_count}. Tentando validação lógica...")
            try:
                dados_processo = json.loads(codigo_json)

                # NOVA VALIDAÇÃO: Validação lógica do BPMN (consistência de flows e elementos)
                if validar_bpmn_logicamente(dados_processo):
                    print(f"✅ JSON sintaticamente e logicamente válido encontrado na tentativa {retry_count}. Retornando.\n")
                    return dados_processo, retry_count, generation_time
                else:
                    print(f"❌ JSON sintaticamente válido, mas falhou na validação lógica BPMN na tentativa {retry_count}. Retentando...")
            except json.JSONDecodeError as e:
                print(f"Erro inesperado ao parsear JSON válido: {e}")
            except TypeError as e:
                print(f"❌ Erro de tipo (TypeError) inesperado durante a validação lógica BPMN na tentativa {retry_count}: {e}. Retentando...")
        else:
            print(f"❌ JSON inválido na tentativa {retry_count}. Conteúdo bruto: {cleaned_output[:500]}...")

    print(f"\n⛔ Não foi possível gerar um JSON sintaticamente e logicamente válido após {max_attempts} tentativas.")
    return None, retry_count, generation_time

In [ ]:
import warnings
import os
import re
import json
import xml.etree.ElementTree as ET
from google.colab import files

# Silencia avisos visuais do Hugging Face
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*max_new_tokens.*")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

# 1. Entrada de texto do usuário
# sua_nova_descricao = (
#     "O processo começa quando o cliente envia uma solicitação de suporte. En seguida, o analista faz a triagem do chamado. "
#     "Depois, ocorre uma tomada de decisão: se o problema for simples, o analista resolve o chamado e o processo termina. "
#     "Se o problema for complexo, o chamado é encaminhado para a equipe de especialistas, que resolve o problema e finaliza o processo."
# )
sua_nova_descricao = coherence_df.loc[0, 'text_description']


# 2. PROMPT CORRIGIDO: Instruções explícitas de formato e esquema
# prompt_sistema = """Você é um tradutor BPMN estrito. Sua única função é mapear o texto do usuário no esquema JSON exato fornecido abaixo.

# REGRAS DE CONFORMIDADE:
# 1. A lista 'elements' DEVE conter apenas nós físicos (startEvent, userTask, serviceTask, exclusiveGateway, endEvent). Nunca coloque 'sequenceFlow' aqui.
# 2. A lista 'sequence_flows' DEVE mapear todas as conexões usando EXATAMENTE as chaves 'id', 'from' e 'to'. Nunca use 'source' ou 'target'.
# 3. Garanta que TODO elemento criado possua pelo menos uma conexão vinculada a ele (evite nós órfãos).

# Retorne APENAS o JSON estruturado abaixo, sem textos adicionais:
# {
#   "process_name": "Nome_Do_Processo",
#   "elements": [
#     {"id": "id_do_no", "type": "tipo_do_no", "name": "Nome amigável"}
#   ],
#   "sequence_flows": [
#     {"id": "id_da_seta", "from": "id_origem", "to": "id_destino"}
#   ]
# }"""

prompt_sistema = """You are a strict BPMN translator. Your sole function is to map the user's text into the exact JSON schema provided below.

COMPLIANCE RULES:
1. The 'elements' list MUST contain only physical nodes (startEvent, userTask, serviceTask, exclusiveGateway, endEvent). Never place 'sequenceFlow' here.
2. The 'sequence_flows' list MUST map all connections using EXACTLY the keys 'id', 'from', and 'to'. Never use 'source' or 'target'.
3. Ensure that EVERY created element has at least one connection linked to it (avoid orphaned nodes).

Return ONLY the structured JSON below, with no additional text:
{
  "process_name": "Process_Name",
  "elements": [
    {"id": "node_id", "type": "node_type", "name": "Friendly name"}
  ],
  "sequence_flows": [
    {"id": "flow_id", "from": "source_id", "to": "destination_id"}
  ]
}"""


# 3. Formatação correta usando o chat template do modelo (essencial para Unsloth/Llama 3)
mensagens = [
    {"role": "system", "content": prompt_sistema},
    {"role": "user", "content": f"Converta este processo:\n{sua_nova_descricao}"}
]

inputs = tokenizer.apply_chat_template(
    mensagens,
    tokenize=True,
    add_generation_prompt=True, # Força o modelo a começar a responder como 'assistant'
    return_tensors="pt"
).to("cuda")

# Cálculos de tokens do Unsloth
input_prompt_token_length = inputs.shape[1]
print(f"Comprimento do prompt de entrada em tokens: {input_prompt_token_length}")
max_new_tokens_permitido = max_seq_length - input_prompt_token_length
print(f"Máximo de novos tokens permitidos: {max_new_tokens_permitido}")

Comprimento do prompt de entrada em tokens: 321
Máximo de novos tokens permitidos: 7871


In [ ]:
# Chamada da nova função para gerar JSON válido com retentativas
json_output, attempts, generation_time = run_model_until_valid_json(
    model=model,
    tokenizer=tokenizer,
    inputs=inputs,
    max_new_tokens_permitido=max_new_tokens_permitido,
    input_prompt_token_length=input_prompt_token_length,
    do_sample=False,             # Mantendo a amostragem para maior variação
    repetition_penalty=1,
    use_cache=False,
    max_attempts=5              # Número máximo de tentativas
)

if json_output:
    dados_processo = json_output
    print(f"\n✅ JSON válido obtido após {attempts} tentativa(s).")
else:
    print(f"\n❌ Não foi possível obter um JSON válido após {attempts} tentativa(s).")
    dados_processo = None # Ou trate o erro conforme sua necessidade

Both `max_new_tokens` (=7871) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⚡ Tentativa 1 de 5: Gerando texto...
Tempo de geração do modelo: 362.72 segundos
✅ JSON sintaticamente válido encontrado na tentativa 1. Tentando validação lógica...
✅ JSON sintaticamente e logicamente válido encontrado na tentativa 1. Retornando.


✅ JSON válido obtido após 1 tentativa(s).


In [ ]:
import xml.etree.ElementTree as ET_lxml
import spacy
from sentence_transformers import SentenceTransformer, util
import networkx as nx

# Carregando os dados de referência e o XML gerado
bpmn_xml_string = gerar_bpmn_definitivo(dados_processo)

# XML gerado pelo modelo (já disponível na variável bpmn_xml_string)
xml_gerado = bpmn_xml_string

# XML de referência do dataset (primeiro registro)
xml_referencia = coherence_df.loc[0, 'bpmn_xml']

# Descrição original do processo (para coerência de requisitos)
texto_referencia = sua_nova_descricao

print("Dados de referência e XML gerado carregados para as métricas.")

# Executa todas as métricas modularizadas

# Certifica-se de que nlp e model_st estão carregados
nlp = spacy.load('en_core_web_sm') # Carrega o modelo spaCy
model_st = SentenceTransformer('paraphrase-MiniLM-L6-v2') # Carrega o SentenceTransformer

# 2. Cobertura de Requisitos (Semantic Similarity)
semantic_similarity_score = calculate_requirements_coverage_metric(texto_referencia, xml_gerado, nlp, model_st)
print(f"Cobertura de Requisitos (Similaridade Semântica): {semantic_similarity_score:.2f}%")

# 3. Complexidade Estrutural (Graph Analysis)
structural_complexity_score = calculate_structural_complexity_metric(xml_gerado)
print(f"Complexidade Estrutural (Densidade do Grafo): {structural_complexity_score:.2f}%")

# 4. Graph Edit Distance (GED)
ged_similarity_score = calculate_ged_metric(xml_gerado, xml_referencia)
print(f"Graph Edit Distance (Similaridade GED Heurística): {ged_similarity_score:.2f}%")

XML BPMN gerado com IDs compatíveis. Retornando string...
Dados de referência e XML gerado carregados para as métricas.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cobertura de Requisitos (Similaridade Semântica): 73.67%
Complexidade Estrutural (Densidade do Grafo): 20.00%
Graph Edit Distance (Similaridade GED Heurística): 38.30%


In [ ]:
import pandas as pd

# Calcula a Taxa de Erro Sintático com base nas tentativas de geração de JSON
# 'attempts' é o número de tentativas até um JSON válido ser encontrado (ou max_attempts se falhar)
# 'json_output' é None se nenhum JSON válido foi gerado

if json_output: # Um JSON válido foi encontrado
    failed_attempts = attempts - 1
    total_attempts_made = attempts
    if total_attempts_made > 0:
        taxa_erro_sintatico = (failed_attempts / total_attempts_made) * 100
    else: # Caso teoricamente não alcançado se attempts > 0
        taxa_erro_sintatico = 0.0
else: # Nenhum JSON válido foi encontrado após max_attempts
    taxa_erro_sintatico = 100.0 # Todos os attempts falharam

# Coleta os valores das métricas
metrics_data = {
    'Métrica': [
        'Taxa de Erro Sintático (JSON Gerado)', # Renomeado para maior clareza
        'Cobertura de Requisitos (Similaridade Semântica)',
        'Complexidade Estrutural (Densidade do Grafo)',
        'Graph Edit Distance (Similaridade GED Heurística)',
        'Tempo de Execução (Geração JSON)'
    ],
    'Valor': [
        f'{taxa_erro_sintatico:.2f}%',
        f'{semantic_similarity_score:.2f}%',
        f'{structural_complexity_score:.2f}%',
        f'{ged_similarity_score:.2f}%',
        f'{generation_time:.2f}s'
    ]
}

# Cria o DataFrame e exibe
df_metrics = pd.DataFrame(metrics_data)
display(df_metrics)

,Métrica,Valor
0,Taxa de Erro Sintático (JSON Gerado),0.00%
1,Cobertura de Requisitos (Similaridade Semântica),73.67%
2,Complexidade Estrutural (Densidade do Grafo),20.00%
3,Graph Edit Distance (Similaridade GED Heurística),38.30%
4,Tempo de Execução (Geração JSON),362.72s
